# 실험 4. LightGBM 하이퍼파라미터

- 기준: `exp01`의 기준 모델 (`learning_rate=0.05`, `num_leaves=63`, OOF AUC 0.95866)
- 방법: 파라미터 하나(또는 한 묶음)만 바꾸고 같은 Fold로 OOF AUC 비교

## 환경설정

In [1]:
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
START_TIME = time.time()

## 데이터 경로 탐색

In [2]:
REQUIRED_FILES = ["train.csv", "test.csv", "sample_submission.csv"]


def has_all_files(folder: Path) -> bool:
    return all((folder / name).is_file() for name in REQUIRED_FILES)

# 크로스플랫폼 경로 자동 해결

def find_data_dir() -> Path:
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates = sorted({p.parent for p in kaggle_root.glob("**/train.csv") if has_all_files(p.parent)})
    else:
        local = [p.resolve() for p in (Path("data"), Path("../data")) if has_all_files(p)]
        candidates = list(dict.fromkeys(local))  # drop duplicates, keep order

    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            f"데이터 폴더를 찾지 못했습니다. {REQUIRED_FILES} 세 파일이 모두 있는 폴더가 필요합니다.\n"
            "- Kaggle: 오른쪽 Input 패널에서 대회 데이터를 추가하세요.\n"
            "- 로컬: 노트북과 같은 위치의 data/ 폴더에 파일을 넣으세요."
        )
    raise RuntimeError(
        "데이터 폴더 후보가 여러 개입니다. 사용할 폴더를 DATA_DIR에 직접 지정하세요:\n"
        + "\n".join(f"  - {c}" for c in candidates)
    )


DATA_DIR = find_data_dir()
print("DATA_DIR =", DATA_DIR)

DATA_DIR = /Users/youngjin/ai/kaggle_practice/Kaggle_Airline_Satisfaction/data


## 데이터 로드 및 전처리

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

TARGET = "satisfaction"
CAT_COLS = ["Gender", "Customer Type", "Type of Travel", "Class"]

In [4]:
y = train[TARGET].astype(int)
X = train.drop(columns=["id", TARGET]).copy()
X_test = test.drop(columns=["id"]).copy()

for col in CAT_COLS:
    categories = sorted(train[col].unique())
    X[col] = pd.Categorical(X[col], categories=categories)
    X_test[col] = pd.Categorical(X_test[col], categories=categories)

assert list(X.columns) == list(X_test.columns)
print("X      :", X.shape)
print("X_test :", X_test.shape)
print("y mean :", round(y.mean(), 4))

X      : (699635, 21)
X_test : (299844, 21)
y mean : 0.4436


## 교차검증 함수

In [5]:
LGB_PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "seed": SEED,
    "verbose": -1,
}
N_FOLDS = 5


def run_cv(X, X_test, params=LGB_PARAMS):
    """Stratified 5-fold LightGBM; returns the OOF AUC and the averaged test prediction."""
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof_pred = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        dtrain = lgb.Dataset(X.iloc[tr_idx], label=y.iloc[tr_idx])
        dvalid = lgb.Dataset(X.iloc[va_idx], label=y.iloc[va_idx], reference=dtrain)

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=5000,
            valid_sets=[dvalid],
            callbacks=[lgb.early_stopping(100, verbose=False)],
        )

        oof_pred[va_idx] = model.predict(X.iloc[va_idx], num_iteration=model.best_iteration)
        test_pred += model.predict(X_test, num_iteration=model.best_iteration) / N_FOLDS
        print(f"Fold {fold}: AUC = {roc_auc_score(y.iloc[va_idx], oof_pred[va_idx]):.5f}   trees = {model.best_iteration}")

    return roc_auc_score(y, oof_pred), test_pred

In [6]:
results = {}

## 기준 모델

In [7]:
auc, _ = run_cv(X, X_test)
results["기준 (LGB_PARAMS)"] = auc
print(f"\nOOF AUC = {auc:.5f}")

Fold 1: AUC = 0.95877   trees = 549


Fold 2: AUC = 0.95759   trees = 293


Fold 3: AUC = 0.95930   trees = 286


Fold 4: AUC = 0.95870   trees = 280


Fold 5: AUC = 0.95902   trees = 256

OOF AUC = 0.95866


## 실험 4-1: learning_rate 0.02

- 이유: 학습률을 낮추면 트리를 더 많이 쌓으며 세밀하게 수렴해 일반화가 좋아질 수 있음 (대신 학습 시간 증가)

In [8]:
params = {**LGB_PARAMS, "learning_rate": 0.02}
auc, _ = run_cv(X, X_test, params)
results["learning_rate 0.02"] = auc
print(f"\nOOF AUC = {auc:.5f}   (기준 대비 {auc - results['기준 (LGB_PARAMS)']:+.5f})")

Fold 1: AUC = 0.95876   trees = 959


Fold 2: AUC = 0.95746   trees = 848


Fold 3: AUC = 0.95927   trees = 591


Fold 4: AUC = 0.95864   trees = 787


Fold 5: AUC = 0.95903   trees = 685

OOF AUC = 0.95862   (기준 대비 -0.00005)


## 실험 4-2: num_leaves 31

- 이유: 트리를 단순하게 만들어 과적합이 줄어드는지 확인

In [9]:
params = {**LGB_PARAMS, "num_leaves": 31}
auc, _ = run_cv(X, X_test, params)
results["num_leaves 31"] = auc
print(f"\nOOF AUC = {auc:.5f}   (기준 대비 {auc - results['기준 (LGB_PARAMS)']:+.5f})")

Fold 1: AUC = 0.95863   trees = 894


Fold 2: AUC = 0.95722   trees = 650


Fold 3: AUC = 0.95915   trees = 754


Fold 4: AUC = 0.95848   trees = 1020


Fold 5: AUC = 0.95876   trees = 570

OOF AUC = 0.95844   (기준 대비 -0.00022)


## 실험 4-3: num_leaves 127

- 이유: 반대로 표현력을 키웠을 때 상호작용을 더 잡아내는지 확인

In [10]:
params = {**LGB_PARAMS, "num_leaves": 127}
auc, _ = run_cv(X, X_test, params)
results["num_leaves 127"] = auc
print(f"\nOOF AUC = {auc:.5f}   (기준 대비 {auc - results['기준 (LGB_PARAMS)']:+.5f})")

Fold 1: AUC = 0.95887   trees = 266


Fold 2: AUC = 0.95782   trees = 156


Fold 3: AUC = 0.95944   trees = 148


Fold 4: AUC = 0.95875   trees = 188


Fold 5: AUC = 0.95888   trees = 189

OOF AUC = 0.95874   (기준 대비 +0.00008)
